<a href="https://colab.research.google.com/github/Faisaleka21/Machine_Learning/blob/main/Perceptron%20and%20Multilayer%20Peceptron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Memuat dataset
df = pd.read_csv('https://raw.githubusercontent.com/febbisena/DataMining/refs/heads/main/dataset_tahu_berformalin.csv')
df

In [ ]:
# Eksplorasi Data
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe().T

In [ ]:
# Pra-pemrosesan Data
df = df.drop(['Temperature(C)'], axis=1)

In [ ]:
# Pembagian Fitur (X) dan Target (y)
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [ ]:
X.tail()

In [ ]:
# Menampilkan 5 data terakhir dari target
y.tail()

In [ ]:
# Menghitung jumlah label (untuk melihat keseimbangan data)
print(y.value_counts())

In [ ]:
# Membagi data menjadi data latih (train) dan data uji (test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data train : ", len(X_train))
print("Data test  : ", len(X_test))

In [ ]:

# Normalisasi data menggunakan MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_test_scaled

In [ ]:
# Membangun arsitektur model Neural Network
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [ ]:

# Menampilkan ringkasan arsitektur model
model.summary()

In [ ]:

# Kompilasi model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:

# Pengaturan Early Stopping untuk mencegah overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [ ]:

# Melatih model (Training)
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=1000,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Visualisasi Loss dan Accuracy per Epoch
plt.figure(figsize=(6, 8))

In [ ]:
# Grafik Loss
plt.subplot(2, 1, 1)
plt.plot(history.history['loss'], label='Loss (Training)')
plt.plot(history.history['val_loss'], label='Loss (Validation)')
plt.title('Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

In [ ]:

# Grafik Accuracy
plt.subplot(2, 1, 2)
plt.plot(history.history['accuracy'], label='Accuracy (Training)')
plt.plot(history.history['val_accuracy'], label='Accuracy (Validation)')
plt.title('Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:

# Melakukan prediksi pada data uji
y_pred = (model.predict(X_test_scaled) > 0.5).astype("int32")

In [ ]:

# Membuat Confusion Matrix
class_names = y.unique()
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.set(font_scale=1.2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Menampilkan Classification Report
class_report = classification_report(y_test, y_pred, digits=4)
print("Classification Report :\n", class_report)

In [ ]:

# Menampilkan Accuracy Score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy * 100:.2f}%")

In [ ]:

# Membuat DataFrame untuk hasil pengujian
test_df = pd.DataFrame(X_test, columns=X.columns)
test_df['Actual Class'] = y_test.values
test_df['Predicted Class'] = y_pred

test_df

In [ ]:

# Membuat perbandingan jumlah kelas Aktual vs Prediksi
comparison_df = test_df.groupby('Actual Class').size().to_frame('Actual')
comparison_df['Predicted'] = test_df.groupby('Predicted Class').size()

# Mengisi nilai kosong dengan 0 dan mengubah tipe data ke integer
comparison_df.fillna(0, inplace=True)
comparison_df = comparison_df.astype(int)

# Visualisasi Bar Chart Perbandingan Aktual vs Prediksi
ax = comparison_df.plot(kind='bar', figsize=(8, 8), color=['skyblue', 'green'])

# Menambahkan label angka di atas bar
ax.bar_label(ax.containers[0], fmt='%d', padding=3)
ax.bar_label(ax.containers[1], fmt='%d', padding=3)

plt.title('Actual vs Predicted Class Counts')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:

# Menyimpan hasil prediksi ke file Excel
test_df.to_excel('MLP_Data Test & Prediction.xlsx', index=False)

In [ ]:

# Menyimpan model yang sudah dilatih
model.save('Model MLP.h5')